# Mini-projet de synthèse — cumulatif S1 → S4

**Contexte.** Une petite enseigne te confie deux sources :
- un **catalogue produits** au format **API JSON** (`data/catalogue_api.json`) — chaque produit a un bloc `fournisseur` **imbriqué** ;
- ses **ventes** sur deux mois, en deux fichiers (`data/ventes_mois1.csv`, `data/ventes_mois2.csv`).

**Objectif (ce que tu dois en retenir).** Dérouler un mini-pipeline **de bout en bout** en combinant
**tout depuis la S1** : extraire (JSON + CSV), réunir et **joindre** les sources sans rien perdre,
enrichir (`apply`, gestion des valeurs manquantes), analyser (`groupby`, tableau croisé), encapsuler
dans une **classe**, interroger en **SQL** (CTE + fonction de fenêtrage), et conclure. C'est ta
répétition générale avant les vrais projets (S5).

**Méthode.** Besoins métier — **à toi de choisir tes outils**. Vocabulaire : *trier* = mettre dans un
ordre ; *classer* = attribuer un rang. Termine par une **synthèse écrite**.

## Partie A — Extraire les données (S1 + S4)
1. Charge le **catalogue** depuis le JSON en un tableau où les infos du fournisseur sont de **vraies colonnes** (pas un dict dans une case).
2. Réunis les ventes des **deux mois** en un **seul** tableau. Combien de ventes en tout ?

In [21]:
# ton code
# --- CELLULE FOURNIE : on charge la "réponse d'API" depuis le fichier ---
import json, pandas as pd

with open("data/catalogue_api.json", encoding="utf-8") as f:
    reponse_api = json.load(f)          # en vrai : reponse_api = requests.get(url).json()
    
print(type(reponse_api), "| nb éléments:", len(reponse_api))
reponse_api[0]                          # aperçu du 1er élément (structure imbriquée)

catalogue = pd.json_normalize(reponse_api, sep="_")
display(catalogue)

df_mois1 = pd.read_csv("data/ventes_mois1.csv")
display(df_mois1.head())

df_mois2 = pd.read_csv("data/ventes_mois2.csv")
display(df_mois2.head())

<class 'list'> | nb éléments: 12


,produit_id,nom,categorie,prix,fournisseur_nom,fournisseur_pays
0,P01,Produit 1,Mobilier,331.53,AccessOne,Espagne
1,P02,Produit 2,Mobilier,120.93,BureauPlus,Allemagne
2,P03,Produit 3,Bureautique,270.67,MobiPro,Italie
3,P04,Produit 4,Bureautique,104.67,AccessOne,Espagne
4,P05,Produit 5,Informatique,243.73,BureauPlus,Allemagne
5,P06,Produit 6,Informatique,127.20,MobiPro,Italie
6,P07,Produit 7,Bureautique,351.73,AccessOne,Espagne
7,P08,Produit 8,Mobilier,75.45,TechDis,France
8,P09,Produit 9,Accessoires,178.02,AccessOne,Espagne
9,P10,Produit 10,Bureautique,386.40,MobiPro,Italie


,vente_id,date,produit_id,quantite,canal
0,V0001,2026-01-03,P09,16,web
1,V0002,2026-01-07,P98,4,boutique
2,V0003,2026-01-06,P11,2,revendeur
3,V0004,2026-01-28,P07,19,web
4,V0005,2026-01-03,P09,15,revendeur


,vente_id,date,produit_id,quantite,canal
0,W0001,2026-02-15,P12,11,web
1,W0002,2026-02-23,P07,15,revendeur
2,W0003,2026-02-14,P07,9,web
3,W0004,2026-02-21,P02,17,boutique
4,W0005,2026-02-06,P12,20,web


In [22]:
df_mois1_2 = pd.concat([df_mois1, df_mois2], ignore_index=True)

print("Nombre de ligne mois 1 : ", len(df_mois1))
print("Nombre de ligne mois 2 : ", len(df_mois2))
print("Nombre de ligne mois 1 & 2 : ", len(df_mois1_2))
display(df_mois1_2)

Nombre de ligne mois 1 :  55
Nombre de ligne mois 2 :  35
Nombre de ligne mois 1 & 2 :  90


,vente_id,date,produit_id,quantite,canal
0,V0001,2026-01-03,P09,16,web
1,V0002,2026-01-07,P98,4,boutique
2,V0003,2026-01-06,P11,2,revendeur
3,V0004,2026-01-28,P07,19,web
4,V0005,2026-01-03,P09,15,revendeur
...,...,...,...,...,...
85,W0031,2026-02-14,P09,2,web
86,W0032,2026-02-22,P12,1,web
87,W0033,2026-02-08,P08,1,revendeur
88,W0034,2026-02-20,P04,12,boutique


## Partie B — Joindre ventes et catalogue (S4 + S2)
Ajoute à chaque vente les informations de son produit (nom, catégorie, prix, fournisseur). **⚠️ Certaines ventes portent un `produit_id` absent du catalogue : arrange-toi pour ne perdre AUCUNE vente.** Combien de ventes n'ont pas de produit au catalogue ? Puis crée une colonne `montant` = quantité × prix.

In [23]:
# ton code
vente_catalogue = pd.merge(left=df_mois1_2, right=catalogue, on="produit_id", how="left")
display(vente_catalogue)
print(vente_catalogue.isna().sum()) # 12 lignes avec des valeurs manquantes

vente_catalogue["montant"] = vente_catalogue["quantite"] * vente_catalogue["prix"]
display(vente_catalogue.head())

,vente_id,date,produit_id,quantite,canal,nom,categorie,prix,fournisseur_nom,fournisseur_pays
0,V0001,2026-01-03,P09,16,web,Produit 9,Accessoires,178.02,AccessOne,Espagne
1,V0002,2026-01-07,P98,4,boutique,NaN,NaN,NaN,NaN,NaN
2,V0003,2026-01-06,P11,2,revendeur,Produit 11,Bureautique,321.86,MobiPro,Italie
3,V0004,2026-01-28,P07,19,web,Produit 7,Bureautique,351.73,AccessOne,Espagne
4,V0005,2026-01-03,P09,15,revendeur,Produit 9,Accessoires,178.02,AccessOne,Espagne
...,...,...,...,...,...,...,...,...,...,...
85,W0031,2026-02-14,P09,2,web,Produit 9,Accessoires,178.02,AccessOne,Espagne
86,W0032,2026-02-22,P12,1,web,Produit 12,Informatique,188.54,TechDis,France
87,W0033,2026-02-08,P08,1,revendeur,Produit 8,Mobilier,75.45,TechDis,France
88,W0034,2026-02-20,P04,12,boutique,Produit 4,Bureautique,104.67,AccessOne,Espagne


vente_id             0
date                 0
produit_id           0
quantite             0
canal                0
nom                 12
categorie           12
prix                12
fournisseur_nom     12
fournisseur_pays    12
dtype: int64


,vente_id,date,produit_id,quantite,canal,nom,categorie,prix,fournisseur_nom,fournisseur_pays,montant
0,V0001,2026-01-03,P09,16,web,Produit 9,Accessoires,178.02,AccessOne,Espagne,2848.32
1,V0002,2026-01-07,P98,4,boutique,NaN,NaN,NaN,NaN,NaN,NaN
2,V0003,2026-01-06,P11,2,revendeur,Produit 11,Bureautique,321.86,MobiPro,Italie,643.72
3,V0004,2026-01-28,P07,19,web,Produit 7,Bureautique,351.73,AccessOne,Espagne,6682.87
4,V0005,2026-01-03,P09,15,revendeur,Produit 9,Accessoires,178.02,AccessOne,Espagne,2670.30


## Partie C — Enrichir et gérer les manques (S4 + S3)
Classe chaque vente dans une **gamme de prix** selon une règle de ton choix (ex. entrée de gamme / standard / premium). Que fais-tu des ventes dont le **prix est inconnu** (produit hors catalogue) ? Justifie.

In [24]:
# ton code


def gamme_prix(prix):
    if prix < 100:
        return "entrée de gamme" 
    if prix >= 100 and prix <= 300:
        return "standard"
    if prix > 300:
        return "premium"
    else:
        return "Produit hors catalogue"


vente_catalogue["gamme_prix"] = vente_catalogue["prix"].apply(gamme_prix)
display(vente_catalogue)

# On a des données de venytes mais on a un client qui n'est pas dans le catalogue, on ne va pas supprimer la ligne car il y a quand même des ventes mais on n'a juste pas les infos j'ai donc 
# pris la décision de mettre le message "ID CLIENT INCONNU : Prix inconnu et quantité inconnu" pour chacun des clients sans id_client dans le catalogue.

,vente_id,date,produit_id,quantite,canal,nom,categorie,prix,fournisseur_nom,fournisseur_pays,montant,gamme_prix
0,V0001,2026-01-03,P09,16,web,Produit 9,Accessoires,178.02,AccessOne,Espagne,2848.32,standard
1,V0002,2026-01-07,P98,4,boutique,NaN,NaN,NaN,NaN,NaN,NaN,Produit hors catalogue
2,V0003,2026-01-06,P11,2,revendeur,Produit 11,Bureautique,321.86,MobiPro,Italie,643.72,premium
3,V0004,2026-01-28,P07,19,web,Produit 7,Bureautique,351.73,AccessOne,Espagne,6682.87,premium
4,V0005,2026-01-03,P09,15,revendeur,Produit 9,Accessoires,178.02,AccessOne,Espagne,2670.30,standard
...,...,...,...,...,...,...,...,...,...,...,...,...
85,W0031,2026-02-14,P09,2,web,Produit 9,Accessoires,178.02,AccessOne,Espagne,356.04,standard
86,W0032,2026-02-22,P12,1,web,Produit 12,Informatique,188.54,TechDis,France,188.54,standard
87,W0033,2026-02-08,P08,1,revendeur,Produit 8,Mobilier,75.45,TechDis,France,75.45,entrée de gamme
88,W0034,2026-02-20,P04,12,boutique,Produit 4,Bureautique,104.67,AccessOne,Espagne,1256.04,standard


## Partie D — Analyser (S3 + S4)
1. Le **CA par catégorie**, trié de la plus rentable à la moins rentable.
2. Le **CA par fournisseur**.
3. Un **tableau croisé** : catégorie (lignes) × canal de vente (colonnes), rempli par le CA total.

In [25]:
# ton code
ca_categorie = vente_catalogue.groupby("categorie")["montant"].sum()
display(ca_categorie.sort_values(ascending=False))

ca_fournisseur = vente_catalogue.groupby("fournisseur_nom")["montant"].sum()
display(ca_fournisseur.sort_values())

ca_canal_croise = vente_catalogue.pivot_table(index="categorie", columns="canal", values="montant", aggfunc="sum", margins=True, margins_name="ca_total")
display(ca_canal_croise.sort_values(by="ca_total", ascending=True))

categorie
Bureautique     117477.28
Informatique     48601.62
Mobilier         20788.41
Accessoires       9791.10
Name: montant, dtype: float64

fournisseur_nom
BureauPlus    20919.64
TechDis       26209.10
AccessOne     58181.86
MobiPro       91347.81
Name: montant, dtype: float64

canal,boutique,revendeur,web,ca_total
categorie,,,,
Accessoires,3026.34,3560.40,3204.36,9791.10
Mobilier,7964.94,2219.07,10604.40,20788.41
Informatique,16646.39,13212.93,18742.30,48601.62
Bureautique,25933.89,56399.88,35143.51,117477.28
ca_total,53571.56,75392.28,67694.57,196658.41


## Partie E — Encapsuler dans une classe (S3 — OOP)
Écris une classe `RapportVentes` qui reçoit **à sa création** le tableau des ventes (avec `montant`), et propose des méthodes qui **renvoient** (⚠️ `return`, pas `print`) :
1. le **CA total** ;
2. le **CA par catégorie** ;
3. le **produit n°1** en CA (son nom).

Puis crée une instance et appelle les trois méthodes.

In [26]:
# ton code : la classe RapportVentes
class RapportVente:
    def __init__(self, tableau_vente):
        self.tableau_vente = tableau_vente
    
    def ca_total(self):
        return self.tableau_vente["montant"].sum()
    
    def ca_par_categorie(self):
        return self.tableau_vente.groupby("categorie")["montant"].sum()
    
    def best_ca_produit(self):
        ca_produit = self.tableau_vente.groupby("nom")["montant"].sum().sort_values(ascending=False)
        best_product = ca_produit.head(1)
        return best_product.index[0]

In [27]:
# ton code : instance + tests
v = RapportVente(vente_catalogue)
ca_tot = v.ca_total()
print("CA total : \n   - ", ca_tot)

ca_categorie = v.ca_par_categorie()
print("\n\nCA par catégorie : ")
display(ca_categorie.sort_values(ascending=True))

best_ca = v.best_ca_produit()
print("Meilleur produit : \n    - ", best_ca)

CA total : 
   -  196658.41


CA par catégorie : 


categorie
Accessoires       9791.10
Mobilier         20788.41
Informatique     48601.62
Bureautique     117477.28
Name: montant, dtype: float64

Meilleur produit : 
    -  Produit 11


## Partie F — En SQL (S2 + S3 + S4)
La cellule ci-dessous est **fournie** : elle reconstruit les données dans une base SQLite (table `faits` = les ventes enrichies, table `produits` = le catalogue) et te donne `q("...")`. Exécute-la, puis écris tes requêtes.

In [28]:
# --- CELLULE FOURNIE : rien à modifier, exécute-la ---
import json, sqlite3, pandas as pd
_cat = pd.json_normalize(json.load(open("data/catalogue_api.json", encoding="utf-8")), sep="_")
_ventes = pd.concat([pd.read_csv("data/ventes_mois1.csv"), pd.read_csv("data/ventes_mois2.csv")], ignore_index=True)
_faits = _ventes.merge(_cat, on="produit_id", how="left")
_faits["montant"] = _faits["quantite"] * _faits["prix"]
_con = sqlite3.connect(":memory:")
_faits.to_sql("faits", _con, index=False, if_exists="replace")
_cat.to_sql("produits", _con, index=False, if_exists="replace")
def q(sql):
    return pd.read_sql(sql, _con)
print("Tables prêtes : faits", _faits.shape[0], "lignes | produits", _cat.shape[0], "lignes")

Tables prêtes : faits 90 lignes | produits 12 lignes


**F.1** — Dans **chaque catégorie**, quel produit rapporte le plus de CA ? Numérote les produits par CA au sein de leur catégorie et ne garde que le premier. (fonction de fenêtrage + CTE)

In [29]:
# q("""
#   WITH ...
# """)


**F.2** — Une question de ton choix mêlant **jointure et/ou sous-requête** (ex. les fournisseurs dont le CA total dépasse un seuil).

In [30]:
# q("""
#   ...
# """)


## Partie G — Synthèse
En 4 à 6 phrases : catégories/fournisseurs qui pèsent le plus, canal dominant, produit phare, et **limites des données** (les ventes hors catalogue). C'est la partie la plus importante.

_Écris ta synthèse ici._